**Student:** Shelia Howard  
**Course:** DSC630-T302 Predictive Analytics (2267-1)  
**Assignment:** Final Project Code Notebook  
**Purpose:** Build and evaluate reproducible classification models for preventive outreach prioritization. This is a proof of concept, not a diagnostic system.   
**Dataset(s):** `healthcare-dataset-stroke-data.csv`

# Predicting Stroke Risk for Preventive Care Outreach

## Introduction

This project evaluates whether demographic, medical, and lifestyle variables from a public stroke dataset can support preventive care outreach prioritization. The analysis is designed as a course-level proof of concept and is not intended to diagnose stroke, determine treatment, or make benefit or eligibility decisions. The analysis uses the Stroke Prediction Dataset published on Kaggle by Fedesoriano (n.d.), which contains 5,110 records and a binary recorded-stroke outcome.

Because recorded stroke cases represent only a small percentage of the dataset, model evaluation emphasizes recall, precision, F1 score, ROC-AUC, PR-AUC, confusion-matrix results, and operational workload rather than accuracy alone. A majority-class benchmark is compared with class-weighted logistic regression, decision tree, random forest, and histogram gradient boosting models.

The primary objective is to identify a model that can detect a meaningful share of the recorded stroke cases while remaining interpretable and producing a review workload that can be understood by healthcare decision makers.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split
)
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
DATA_PATH = Path('healthcare-dataset-stroke-data.csv')

## 1. Load and validate the data
Only the top records are displayed to keep the submitted code PDF short.

In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(df.head())

profile = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2)
})
display(profile)
print('Exact duplicate rows:', df.duplicated().sum())
print('Duplicate rows excluding id:', df.drop(columns=['id']).duplicated().sum())
display(df['stroke'].value_counts().rename_axis('stroke').to_frame('count').assign(percent=lambda x: (x['count']/len(df)*100).round(2)))

Dataset shape: 5,110 rows x 12 columns


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


,dtype,missing,missing_pct
id,int64,0,0.00
gender,str,0,0.00
age,float64,0,0.00
hypertension,int64,0,0.00
heart_disease,int64,0,0.00
ever_married,str,0,0.00
work_type,str,0,0.00
Residence_type,str,0,0.00
avg_glucose_level,float64,0,0.00
bmi,float64,201,3.93


Exact duplicate rows: 0
Duplicate rows excluding id: 0


,count,percent
stroke,,
0,4861,95.13
1,249,4.87


## 2. Descriptive outcome comparisons

The following summaries quantify the preliminary relationships identified during exploratory analysis. These are descriptive associations within this dataset and should not be interpreted as causal effects.

In [3]:
numeric_summary = (
    df.groupby('stroke')[
        ['age', 'avg_glucose_level', 'bmi']
    ]
    .mean()
    .round(2)
)

display(numeric_summary)

for variable in [
    'hypertension',
    'heart_disease',
    'smoking_status',
    'work_type'
]:
    outcome_rates = (
        df.groupby(variable)['stroke']
        .agg(
            Records='size',
            Stroke_Cases='sum',
            Stroke_Rate='mean'
        )
    )

    outcome_rates['Stroke_Rate'] = (
        outcome_rates['Stroke_Rate'] * 100
    ).round(2)

    print(f"\nRecorded stroke rates by {variable}:")
    display(outcome_rates)

,age,avg_glucose_level,bmi
stroke,,,
0,41.97,104.80,28.82
1,67.73,132.54,30.47



Recorded stroke rates by hypertension:


,Records,Stroke_Cases,Stroke_Rate
hypertension,,,
0,4612,183,3.97
1,498,66,13.25



Recorded stroke rates by heart_disease:


,Records,Stroke_Cases,Stroke_Rate
heart_disease,,,
0,4834,202,4.18
1,276,47,17.03



Recorded stroke rates by smoking_status:


,Records,Stroke_Cases,Stroke_Rate
smoking_status,,,
Unknown,1544,47,3.04
formerly smoked,885,70,7.91
never smoked,1892,90,4.76
smokes,789,42,5.32



Recorded stroke rates by work_type:


,Records,Stroke_Cases,Stroke_Rate
work_type,,,
Govt_job,657,33,5.02
Never_worked,22,0,0.00
Private,2925,149,5.09
Self-employed,819,65,7.94
children,687,2,0.29


## 3. Categorical Data Review

The categorical variables are reviewed to confirm the observed category levels before preprocessing. This check helps verify categories such as the rare `Other` gender value and the `Unknown` smoking-status category so they can be handled consistently during modeling.

In [4]:
# Review categorical levels to confirm observed categories.
categorical_levels = {
    column: sorted(df[column].astype(str).unique())
    for column in df.select_dtypes(include='object').columns
}

for column, levels in categorical_levels.items():
    print(f"{column}: {levels}")

gender: ['Female', 'Male', 'Other']
ever_married: ['No', 'Yes']
work_type: ['Govt_job', 'Never_worked', 'Private', 'Self-employed', 'children']
Residence_type: ['Rural', 'Urban']
smoking_status: ['Unknown', 'formerly smoked', 'never smoked', 'smokes']


## 4. Prepare features and split data

In [5]:
target = 'stroke'

X = df.drop(columns=['id', target])
y = df[target].astype(int)

numeric_features = [
    'age',
    'hypertension',
    'heart_disease',
    'avg_glucose_level',
    'bmi'
]

categorical_features = [
    'gender',
    'ever_married',
    'work_type',
    'Residence_type',
    'smoking_status'
]

# Numeric preprocessing for logistic regression:
# median imputation followed by standardization.
numeric_scaled = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Numeric preprocessing for tree-based models:
# median imputation only.
numeric_unscaled = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Preserve categorical values such as Unknown smoking status.
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocess_scaled = ColumnTransformer(transformers=[
    ('numeric', numeric_scaled, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])

preprocess_tree = ColumnTransformer(transformers=[
    ('numeric', numeric_unscaled, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print(
    'Training set:',
    X_train.shape,
    'positive rate:',
    f'{y_train.mean():.2%}'
)

print(
    'Test set:',
    X_test.shape,
    'positive rate:',
    f'{y_test.mean():.2%}'
)

Training set: (4088, 10) positive rate: 4.87%
Test set: (1022, 10) positive rate: 4.89%


## 5. Build and Tune Candidate Models

In [6]:
# ============================================================
# Build and Tune Candidate Models
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    'recall': 'recall',
    'precision': 'precision',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision'
}

# Majority-class benchmark
majority_model = Pipeline(steps=[
    ('preprocess', preprocess_tree),
    ('model', DummyClassifier(strategy='most_frequent'))
])

# Interpretable class-weighted logistic regression
logistic_model = Pipeline(steps=[
    ('preprocess', preprocess_scaled),
    ('model', LogisticRegression(
        class_weight='balanced',
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

# ------------------------------------------------------------
# Tune Decision Tree
# ------------------------------------------------------------

tree_search = GridSearchCV(
    estimator=Pipeline(steps=[
        ('preprocess', preprocess_tree),
        ('model', DecisionTreeClassifier(
            class_weight='balanced',
            random_state=RANDOM_STATE
        ))
    ]),
    param_grid={
        'model__max_depth': [3, 5, 7, None],
        'model__min_samples_leaf': [5, 15, 30, 50]
    },
    scoring='average_precision',
    cv=cv,
    n_jobs=-1
)

tree_search.fit(X_train, y_train)

# ------------------------------------------------------------
# Tune Random Forest
# ------------------------------------------------------------

forest_search = GridSearchCV(
    estimator=Pipeline(steps=[
        ('preprocess', preprocess_tree),
        ('model', RandomForestClassifier(
            class_weight='balanced_subsample',
            n_estimators=400,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
    param_grid={
        'model__max_depth': [5, 10, None],
        'model__min_samples_leaf': [2, 5, 10],
        'model__max_features': ['sqrt', 0.7]
    },
    scoring='average_precision',
    cv=cv,
    n_jobs=-1
)

forest_search.fit(X_train, y_train)

# ------------------------------------------------------------
# Tune Histogram Gradient Boosting
# ------------------------------------------------------------

boost_search = GridSearchCV(
    estimator=Pipeline(steps=[
        ('preprocess', preprocess_tree),
        ('model', HistGradientBoostingClassifier(
            class_weight='balanced',
            random_state=RANDOM_STATE
        ))
    ]),
    param_grid={
        'model__learning_rate': [0.03, 0.07, 0.10],
        'model__max_leaf_nodes': [7, 15, 31],
        'model__min_samples_leaf': [20, 40]
    },
    scoring='average_precision',
    cv=cv,
    n_jobs=-1
)

boost_search.fit(X_train, y_train)

# Store the benchmark, baseline, and tuned models.
models = {
    'Majority Benchmark': majority_model,
    'Class-Weighted Logistic Regression': logistic_model,
    'Decision Tree': tree_search.best_estimator_,
    'Random Forest': forest_search.best_estimator_,
    'Histogram Gradient Boosting': boost_search.best_estimator_
}

print("Decision Tree best parameters:")
print(tree_search.best_params_)

print("\nRandom Forest best parameters:")
print(forest_search.best_params_)

print("\nHistogram Gradient Boosting best parameters:")
print(boost_search.best_params_)

Decision Tree best parameters:
{'model__max_depth': 7, 'model__min_samples_leaf': 50}

Random Forest best parameters:
{'model__max_depth': None, 'model__max_features': 0.7, 'model__min_samples_leaf': 10}

Histogram Gradient Boosting best parameters:
{'model__learning_rate': 0.07, 'model__max_leaf_nodes': 15, 'model__min_samples_leaf': 20}


## 6. Cross-Validated Model Comparison

In [7]:
# ============================================================
# Cross-Validated Model Comparison
# ============================================================

cv_rows = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        error_score='raise'
    )

    cv_rows.append({
        'Model': name,
        'Recall': scores['test_recall'].mean(),
        'Precision': scores['test_precision'].mean(),
        'F1': scores['test_f1'].mean(),
        'ROC-AUC': scores['test_roc_auc'].mean(),
        'PR-AUC': scores['test_pr_auc'].mean()
    })

cv_table = (
    pd.DataFrame(cv_rows)
    .sort_values('PR-AUC', ascending=False)
    .reset_index(drop=True)
)

display(cv_table.round(3))

,Model,Recall,Precision,F1,ROC-AUC,PR-AUC
0,Histogram Gradient Boosting,0.542,0.163,0.250,0.831,0.218
1,Random Forest,0.417,0.199,0.269,0.843,0.197
2,Class-Weighted Logistic Regression,0.789,0.133,0.228,0.839,0.193
3,Decision Tree,0.773,0.126,0.216,0.803,0.153
4,Majority Benchmark,0.000,0.000,0.000,0.500,0.049


### Cross-Validation Interpretation

Histogram gradient boosting produced the highest mean cross-validated PR-AUC at 0.218, while random forest produced the highest mean ROC-AUC at 0.843. However, their recall values at the default threshold were substantially lower than those of class-weighted logistic regression and the decision tree. Class-weighted logistic regression achieved a mean recall of 0.789 while maintaining competitive discrimination and greater interpretability.

Because the project is designed for preventive outreach prioritization, model selection is not based on a single performance metric. Recall is especially important because missed positive cases represent patients who would not be prioritized for review, while interpretability is important when model results are used to support human healthcare decisions.

## 7. Holdout-Test Comparison at the Default Threshold

The untouched 20% test set is used to compare the candidate models after model development. Accuracy is reported for completeness but is not used as the primary selection criterion because the positive stroke class is rare. The comparison helps show the tradeoff between identifying more recorded stroke cases and limiting the number of false-positive outreach reviews.

In [8]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test, threshold=0.50):
    fitted_model = clone(model).fit(X_train, y_train)
    probabilities = fitted_model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        'Model': name,
        'Threshold': threshold,
        'Accuracy': accuracy_score(y_test, predictions),
        'Recall': recall_score(y_test, predictions, zero_division=0),
        'Precision': precision_score(y_test, predictions, zero_division=0),
        'F1': f1_score(y_test, predictions, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, probabilities),
        'PR-AUC': average_precision_score(y_test, probabilities),
        'TN': tn,
        'FP': fp,
        'FN': fn,
        'TP': tp,
        'Flagged': fp + tp,
        'Flag Rate': (fp + tp) / len(y_test)
    }


test_rows = []

for name, model in models.items():
    test_rows.append(
        evaluate_model(
            name,
            model,
            X_train,
            y_train,
            X_test,
            y_test
        )
    )

test_results = (
    pd.DataFrame(test_rows)
    .sort_values('PR-AUC', ascending=False)
    .reset_index(drop=True)
)

display(test_results.round(3))

,Model,Threshold,Accuracy,Recall,Precision,F1,ROC-AUC,PR-AUC,TN,FP,FN,TP,Flagged,Flag Rate
0,Class-Weighted Logistic Regression,0.5,0.746,0.80,0.138,0.235,0.844,0.269,722,250,10,40,290,0.284
1,Random Forest,0.5,0.904,0.48,0.250,0.329,0.837,0.241,900,72,26,24,96,0.094
2,Histogram Gradient Boosting,0.5,0.865,0.62,0.207,0.310,0.828,0.224,853,119,19,31,150,0.147
3,Decision Tree,0.5,0.719,0.78,0.124,0.214,0.813,0.183,696,276,11,39,315,0.308
4,Majority Benchmark,0.5,0.951,0.00,0.000,0.000,0.500,0.049,972,0,50,0,0,0.000


### Holdout Comparison Interpretation

The holdout results reinforce the tradeoff between sensitivity and workload. Random forest achieved the highest precision and F1 score among the trained models, but it identified only 24 of the 50 recorded stroke cases. Histogram gradient boosting identified 31 cases, while the decision tree identified 39 but generated the largest review workload.

Class-weighted logistic regression identified 40 of the 50 recorded stroke cases at the default threshold while maintaining the strongest ROC-AUC and PR-AUC on the holdout set. These results support using logistic regression as the primary operational candidate before selecting a final outreach threshold.

## 8. Select Threshold for Logistic Regression

The final operating threshold is selected using out-of-fold predictions from the training data rather than the test outcomes. The project uses a working target of at least 80% recall because missing recorded stroke cases is more consequential for the outreach use case than generating some additional human reviews. The 80% target is a course-project decision and should not be interpreted as a clinically validated standard.

In [9]:
# Generate out-of-fold predicted probabilities using only
# the training data. This prevents the holdout test set from
# influencing threshold selection.
oof_prob = cross_val_predict(
    models["Class-Weighted Logistic Regression"],
    X_train,
    y_train,
    cv=cv,
    method="predict_proba"
)[:, 1]

# Calculate precision, recall, and the corresponding thresholds
# from the out-of-fold training predictions.
precision, recall, thresholds = precision_recall_curve(
    y_train,
    oof_prob
)

# precision_recall_curve returns one more precision/recall value
# than threshold value, so exclude the final precision/recall entry.
threshold_results = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1]
})

# Select only thresholds that meet the project goal
# of at least 80% recall.
eligible_thresholds = threshold_results[
    threshold_results["recall"] >= 0.80
].copy()

# Among thresholds meeting the recall target, choose the one
# with the highest precision. If precision ties, use the higher
# threshold as the tie-breaker.
if not eligible_thresholds.empty:
    chosen = eligible_thresholds.sort_values(
        ["precision", "threshold"],
        ascending=[False, False]
    ).iloc[0]
else:
    # Fallback: if no threshold reaches 80% recall, choose the
    # threshold with recall closest to 80%.
    chosen = threshold_results.iloc[
        (threshold_results["recall"] - 0.80).abs().argmin()
    ]

selected_threshold = float(chosen["threshold"])

print(
    "Training-derived threshold:",
    round(selected_threshold, 3)
)
print(
    "OOF training recall:",
    round(float(chosen["recall"]), 3)
)
print(
    "OOF training precision:",
    round(float(chosen["precision"]), 3)
)

Training-derived threshold: 0.493
OOF training recall: 0.804
OOF training precision: 0.133


## 9. Final holdout evaluation

In [10]:
# Fit a fresh copy of the selected logistic regression model
# using the full training set.
final_model = clone(
    models["Class-Weighted Logistic Regression"]
).fit(X_train, y_train)

# Generate predicted probabilities for the untouched test set.
test_proba = final_model.predict_proba(X_test)[:, 1]

# Apply the threshold selected from out-of-fold training predictions.
test_pred = (test_proba >= selected_threshold).astype(int)

# Calculate the confusion matrix.
tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_pred,
    labels=[0, 1]
).ravel()

# Calculate final evaluation metrics.
final_results = pd.DataFrame({
    "Metric": [
        "Threshold",
        "Accuracy",
        "Recall",
        "Precision",
        "F1 Score",
        "ROC-AUC",
        "PR-AUC",
        "True Negatives",
        "False Positives",
        "False Negatives",
        "True Positives",
        "Total Flagged",
        "Flag Rate"
    ],
    "Result": [
        selected_threshold,
        accuracy_score(y_test, test_pred),
        recall_score(y_test, test_pred, zero_division=0),
        precision_score(y_test, test_pred, zero_division=0),
        f1_score(y_test, test_pred, zero_division=0),
        roc_auc_score(y_test, test_proba),
        average_precision_score(y_test, test_proba),
        tn,
        fp,
        fn,
        tp,
        fp + tp,
        (fp + tp) / len(y_test)
    ]
})

display(final_results.round(3))

,Metric,Result
0,Threshold,0.493
1,Accuracy,0.744
2,Recall,0.800
3,Precision,0.137
4,F1 Score,0.234
5,ROC-AUC,0.844
6,PR-AUC,0.269
7,True Negatives,720.000
8,False Positives,252.000
9,False Negatives,10.000


## 10. Interpretation of Final Results

The majority-class benchmark demonstrates why accuracy alone is not appropriate for this project. Although predicting the majority class can produce approximately 95% accuracy, it does not identify any of the recorded stroke cases.

Class-weighted logistic regression provides the best fit for the stated outreach objective because it reaches the working recall target while remaining interpretable. At the training-selected threshold of approximately 0.493, the model identifies 40 of the 50 recorded stroke cases in the holdout test set and misses 10.

The higher sensitivity creates an operational tradeoff. The model also flags 252 records without a recorded stroke, resulting in 292 total records, or approximately 28.6% of the test population, being routed for review. This means the model should be viewed as a prioritization tool rather than an automated clinical decision system. Whether the threshold is operationally useful would depend on available care-management capacity and the cost of reviewing additional records.

## 11. Conclusion
Class-weighted logistic regression is recommended for the proof of concept because it meets the recall target and remains interpretable. The model is not ready for deployment. External validation, calibration, fairness testing, privacy controls, clinical review, and a limited prospective pilot are required before operational use.